# 02 — Pré-processamento corrigido das entrevistas KYRA — v3

Este notebook reconstrói a base de modelagem a partir da tabela de entrevistas completas (`entrevistas_docs_v2`) e salva todos os outputs dentro de `data/`.

Correções desta versão:

1. Limpeza antes da segmentação de page counters, footers, nomes de arquivo, links, `Meeting Title`, `Transcribed URL`, `Transcribed by [URL]` e marcadores `[URL]`.
2. A base principal fica **somente em português** (`interviews_chunks_modeling.parquet` = `interviews_chunks_modeling_pt.parquet`).
3. O espanhol não desaparece: bases auxiliares `all`, `pt`, `es` e `mixed` continuam sendo salvas.
4. `interviews_tokens_long.parquet` fica alinhado com a base principal em português.
5. Stopwords/fillers foram endurecidos para `text_for_keywords` e `tokens_topic`.
6. Diagnósticos antigos/stale agora são sobrescritos com nomes padronizados, incluindo `08_preprocess_summary.json`, `README_preprocessamento.md`, `02_tokens_limpos_top3000.csv` e `06_tfidf_por_projeto_top50.csv`.

A entrada recomendada para embeddings/clusters é:

```text
data/processed/interviews_chunks_modeling.parquet
```

Coluna de embedding:

```text
text_for_embedding
```

Coluna para keywords/explicação:

```text
text_for_keywords
```


In [1]:
import pandas as pd, numpy as np, re, unicodedata, hashlib, json
from pathlib import Path
from collections import Counter, defaultdict
from datetime import datetime

# ============================================================
# 0. Paths e configuração
# ============================================================
RUN_ID = datetime.now().strftime('%Y%m%d_%H%M%S')

try:
    NOTEBOOK_DIR = Path(__file__).resolve().parent
except NameError:
    NOTEBOOK_DIR = Path.cwd()

PROJECT_DIR = NOTEBOOK_DIR.parent if NOTEBOOK_DIR.name.lower() == 'notebooks' else NOTEBOOK_DIR
DATA_DIR = PROJECT_DIR / 'data'
RAW_DIR = DATA_DIR / 'raw'
SNAPSHOT_DIR = RAW_DIR / 'bq_snapshots'
PROCESSED_DIR = DATA_DIR / 'processed'
DIAG_DIR = DATA_DIR / 'diagnostics' / 'preprocess'
for d in [RAW_DIR, SNAPSHOT_DIR, PROCESSED_DIR, DIAG_DIR]:
    d.mkdir(parents=True, exist_ok=True)

DOCS_TABLE = 'kyra-pml-20261.kyra_base_2026_1.entrevistas_docs_v2'
OLD_CHUNKS_TABLE = 'kyra-pml-20261.kyra_base_2026_1.entrevistas_chunks_v2'  # referência apenas; não é usado para segmentar
USE_BIGQUERY_IF_LOCAL_NOT_FOUND = True

TEXT_COL='texto_analise'; DOC_ID_COL='doc_id'; LANG_COL='idioma'
TARGET_WORDS=250; MIN_WORDS=120; MAX_WORDS=330; HARD_MAX_WORDS=390
MIN_TOPIC_TOKENS=18
ALLOWED_LANG='pt'


def table_to_snapshot_name(table):
    return table.replace('-', '_').replace('.', '__') + '.parquet'


def load_table_or_snapshot(table):
    name = table_to_snapshot_name(table)
    candidates = [SNAPSHOT_DIR / name, RAW_DIR / name, DATA_DIR / name, PROJECT_DIR / name]
    for p in candidates:
        if p.exists():
            print(f'Carregando snapshot local: {p}')
            return pd.read_parquet(p)
    if not USE_BIGQUERY_IF_LOCAL_NOT_FOUND:
        raise FileNotFoundError('Snapshot local não encontrado: ' + '\n'.join(map(str, candidates)))
    try:
        from google.cloud import bigquery
    except ImportError as exc:
        raise ImportError(
            'google-cloud-bigquery não está instalado. Instale com pip install google-cloud-bigquery db-dtypes, '            'ou salve o parquet em data/raw/bq_snapshots.'
        ) from exc
    print(f'Baixando BigQuery: {table}')
    client = bigquery.Client(project=table.split('.')[0])
    df = client.query(f'SELECT * FROM `{table}`').to_dataframe()
    out = SNAPSHOT_DIR / name
    df.to_parquet(out, index=False)
    print(f'Snapshot salvo em: {out}')
    return df


docs_raw = load_table_or_snapshot(DOCS_TABLE)
print('RUN_ID', RUN_ID)
print('PROJECT_DIR', PROJECT_DIR)
print('docs_raw', docs_raw.shape)

required = [DOC_ID_COL, TEXT_COL]
missing = [c for c in required if c not in docs_raw.columns]
if missing:
    raise ValueError(f'Colunas obrigatórias ausentes: {missing}')
if docs_raw[DOC_ID_COL].isna().any():
    raise ValueError('Há doc_id nulo na base de entrevistas completas.')
if docs_raw[DOC_ID_COL].duplicated().any():
    raise ValueError('Há doc_id duplicado na base de entrevistas completas.')

PAGE_COUNTER_LINE_RE = re.compile(r'^\s*\d{1,4}\s*/\s*\d{1,4}\s*$', re.I)
PAGE_COUNTER_INLINE_RE = re.compile(r'(?<!\w)\d{1,4}\s*/\s*\d{1,4}(?!\w)')
META_LINE_RES = [
    re.compile(r'^\s*(meeting\s+title|meeting\s+created\s+at|transcribed\s+url|recording\s+url|download\s+url|file\s*name|filename|arquivo|nome\s+do\s+arquivo)\s*:', re.I),
    re.compile(r'^\s*https?://\S+\s*$', re.I),
    re.compile(r'^\s*[A-Z]{2}[_-]\d{4}[_-].{8,}$', re.I),
    re.compile(r'^\s*[A-Z]{2}[_-]\d{2,4}[_-].*?(?:concorr[êe]ncia|download|anos|ab\d|c\d).*$', re.I),
    re.compile(r'.*\.(?:m4a|mp3|mp4|wav|aac|pdf)\b.*', re.I),
]
INLINE_NOISE_RES = [
    re.compile(r'Meeting\s+Title\s*:[^\n\r]+', re.I),
    re.compile(r'Meeting\s+created\s+at\s*:[^\n\r]+', re.I),
    re.compile(r'Transcribed\s+URL\s*:?\s*\S+', re.I),
    re.compile(r'Transcribed\s+by\s+(?:\[[A-Z]+\]|https?://\S+|\S+)', re.I),
    re.compile(r'Recording\s+URL\s*:?\s*\S+', re.I),
    re.compile(r'Download\s+URL\s*:?\s*\S+', re.I),
    re.compile(r'https?://\S+', re.I),
    re.compile(r'\[URL\]', re.I),
    re.compile(r'\b\S+\.(?:m4a|mp3|mp4|wav|aac|pdf)\b', re.I),
    re.compile(r'\b[A-Z]{2}[_-]\d{4}[_-][A-Za-z0-9_\-À-ÿ\.]+(?:\s*)', re.I),
]
SPEAKER_RE = re.compile(r'(?im)^\s*(Speaker\s*\d+|SPEAKER[_\s]*\d+|Entrevistador(?:a)?|Moderador(?:a)?|Participante\s*\d*|Entrevistad[oa]\s*\d*)\s*(?:[-–—:]\s*(\d{1,2}:\d{2}(?::\d{2})?))?\s*$')
TIMESTAMP_RE = re.compile(r'\b\d{1,2}:\d{2}(?::\d{2})?\b')
EMAIL_RE=re.compile(r'\b[\w.+-]+@[\w-]+(?:\.[\w-]+)+\b')
PHONE_RE=re.compile(r'(?:\+?55\s*)?(?:\(?\d{2}\)?\s*)?\d{4,5}[-\s]?\d{4}')
CPF_RE=re.compile(r'\b\d{3}\.?\d{3}\.?\d{3}-?\d{2}\b')
CEP_RE=re.compile(r'\b\d{5}-?\d{3}\b')

PT_STOP = set('''a à às o os as um uma uns umas de do da dos das em no na nos nas por para pra pro pros pras com sem sobre sob entre contra até desde após antes durante e ou mas porém porque pois que qual quais quem onde quando como quanto quantos quantas se ser sou é era eram foi foram fui somos são estar estou está estão estava estavam este esta esse essa isso isto aquele aquela aquilo meu minha meus minhas teu tua seus suas dele dela deles delas nosso nossa nossos nossas eu você voce vocês voces tu ele ela eles elas me te lhe nos lhes mim ti si aqui aí ai ali lá la agora já ja ainda também tambem muito muita muitos muitas pouco pouca poucos poucas mais menos cada todo toda todos todas outro outra outros outras mesmo mesma mesmos mesmas tal tais tanto tanta tantos tantas então entao né ne néh neh ah tá ta ok bom boa bem só so sim não nao nem algum alguma alguns algumas nenhum nenhuma nada tudo algo caso forma jeito tipo coisa coisas vez vezes anos ano dia dias gente'''.split())
PT_FILLERS = set('''gente acho sabe sabia tipo assim daí dai né ne neh néh ah eh hum hã uhum tá ta tava estavam estar vou vai vamos ia iria queria queria dizer falou fala falar falando falam falei falou olha então entao bom boa bem beleza certo ok obrigada obrigado tchau oi olá ola joia jóia perfeito maravilha super realmente basicamente literalmente enfim nossa entendeu entende entendido colocar deixa deixa eu deixa-me deixa deixa gente consigo consegue conseguia meio pessoa pessoas pessoal menina meninas menino meninos participante participantes entrevistador entrevistadora moderador moderadora pesquisa sessão sessao reunião reuniao gravação gravacao gravar gravado gravada áudio audio video vídeo call zoom teams meet link transcribed url meeting title speaker download m4a mp3 mp4 wav pdf arquivo pagina página paginas páginas'''.split())
ES_STOP = set('''el la los las un una unos unas de del y o pero que quien donde cuando como cuanto si no ni en por para con sin sobre entre desde hasta este esta esto ese esa eso aquel aquella yo tú tu usted ustedes él ella ellos ellas mi mis su sus nuestro nuestra muy más mas menos todo toda todos todas otro otra otros otras entonces porque pues algo nada bueno buena bien gracias hola adios gente creo sabe tipo'''.split())
EN_STOP = set('''the a an and or but if then else when where who what why how is are was were be been being to of in on for with without by from as at this that these those i you he she it we they me my your his her our their not no yes ok meeting title speaker transcribed url download file'''.split())
STOPWORDS_KEYWORDS = PT_STOP | PT_FILLERS | ES_STOP | EN_STOP
# explicit extra removals requested
STOPWORDS_KEYWORDS |= {'no','das','pra','gente','acho','sabe','falou','vou','nao','não'}

PT_LANG_WORDS = set('''que de o a e do da em um para é com não uma os no se na por mais as dos como mas foi ao ele das tem à seu sua ou ser quando muito há nos já está eu também só pelo pela até isso ela entre era depois sem mesmo aos ter seus quem nas me esse eles estão você tinha foram essa num nem suas meu às minha têm numa pelos elas havia seja qual será nós tenho lhe deles essas esses pelas este fosse dele tu te vocês vos lhes meus minhas teu tua teus tuas nosso nossa nossos nossas dela delas esta isto aquilo quê'''.split())
ES_LANG_WORDS = set('''que de la el en y a los se del las por un para con no una su al lo es como más pero sus le ya o este sí porque esta entre cuando muy sin sobre también me hasta hay donde quien desde todo nos durante todos uno les ni contra otros ese eso ante ellos e esto mí antes algunos qué unos yo otro otras otra él tanto esa estos mucho quienes nada muchos cual poco ella estar estas algunas algo nosotros mi mis tú te ti tu tus ellas nosotras vosotros vosotras os mío mía míos mías tuyo tuya tuyos tuyas suyo suya suyos suyas nuestro nuestra nuestros nuestras vuestro vuestra vuestros vuestras esos esas estoy estás está estamos están esté estés estemos estén'''.split())
EN_LANG_WORDS = set('''the of and to in a is that for it on was with as are be this by at from or an have not they you his her we their there what which when who how why can will would should could about into than then them these those'''.split())

PROCEDURAL_PATTERNS = [
    r'\b(vamos|vou)\s+gravar\b', r'\bgrav(ar|ando|ação|acao)\b', r'\bautoriz(a|ação|acao)\b', r'\bconsentimento\b', r'\blgpd\b',
    r'\bpesquisa\b', r'\bestudo\b', r'\bsomos\s+(a\s+)?(kyra|akira|kira)\b', r'\bmeu\s+nome\s+é\b', r'\bmeu\s+nome\s+e\b',
    r'\bobrigad[ao]s?\s+(pela|por)\s+(participação|participacao|presença|presenca)\b', r'\btchau\b', r'\bencerrar\b',
    r'\bmeeting\s+title\b', r'\btranscribed\s+url\b', r'\btranscribed\s+by\b', r'\[url\]', r'\bagradecer\b'
]
PROCEDURAL_RE = re.compile('|'.join(PROCEDURAL_PATTERNS), re.I)


def normalize_ws(s):
    if s is None or (isinstance(s,float) and np.isnan(s)):
        return ''
    s=str(s).replace('\r','\n').replace('\xa0',' ')
    s=re.sub(r'[\t ]+',' ',s)
    s=re.sub(r'\n[ \t]+','\n',s)
    s=re.sub(r'\n{3,}','\n\n',s)
    return s.strip()

def strip_accents(s):
    return ''.join(ch for ch in unicodedata.normalize('NFKD', str(s)) if not unicodedata.combining(ch))

# Normaliza stopwords/fillers para a mesma forma usada em tokenize_keywords.
STOPWORDS_KEYWORDS = {strip_accents(w.lower()) for w in STOPWORDS_KEYWORDS}
STOPWORDS_KEYWORDS |= {strip_accents(w) for w in ['no','das','pra','gente','acho','sabe','falou','vou','nao','não','tem','ter','tenho','tinha','sao','são','ate','até','fazer','faz','fez','sei']}

def md5_short(s,n=16):
    return hashlib.md5(str(s).encode('utf-8')).hexdigest()[:n]

def clean_pre_segmentation(text):
    s=normalize_ws(text)
    cleaned=[]
    for line in s.splitlines():
        line=line.strip()
        if not line:
            cleaned.append('')
            continue
        if PAGE_COUNTER_LINE_RE.match(line):
            continue
        if any(rx.match(line) for rx in META_LINE_RES):
            continue
        for rx in INLINE_NOISE_RES:
            line=rx.sub(' ',line)
        line=PAGE_COUNTER_INLINE_RE.sub(' ',line)
        line=re.sub(r'\s+',' ',line).strip()
        if line:
            cleaned.append(line)
    s='\n'.join(cleaned)
    s=re.sub(r'\n{3,}','\n\n',s)
    return s.strip()

def clean_text_for_embedding(s):
    s=normalize_ws(s)
    for rx in INLINE_NOISE_RES:
        s=rx.sub(' ',s)
    s=PAGE_COUNTER_INLINE_RE.sub(' ',s)
    s=TIMESTAMP_RE.sub(' ',s)
    s=EMAIL_RE.sub(' [EMAIL] ',s)
    s=CPF_RE.sub(' [CPF] ',s)
    s=CEP_RE.sub(' [CEP] ',s)
    s=PHONE_RE.sub(' [PHONE] ',s)
    s=re.sub(r'\s+',' ',s).strip()
    s=re.sub(r'^[\s\-:–—]+','',s)
    return s

def parse_ts(ts):
    if not ts: return None
    try:
        parts=[int(x) for x in ts.split(':')]
        return parts[0]*60+parts[1] if len(parts)==2 else parts[0]*3600+parts[1]*60+parts[2]
    except Exception:
        return None

def split_sentences(text):
    s=clean_text_for_embedding(text)
    if not s: return []
    parts=re.split(r'(?<=[.!?])\s+(?=[A-ZÁÉÍÓÚÂÊÔÃÕÇ0-9])',s)
    out=[]
    for p in parts:
        p=p.strip()
        if len(p.split())>0:
            out.append(p)
    return out if out else [s]

def extract_turns_for_doc(doc_id, text):
    s=clean_pre_segmentation(text)
    matches=list(SPEAKER_RE.finditer(s))
    rows=[]
    if not matches:
        idx=0
        buffer=[]; n=0
        for sent in split_sentences(s):
            w=len(sent.split())
            if buffer and n+w>MAX_WORDS:
                idx+=1
                txt=' '.join(buffer)
                rows.append({'doc_id':doc_id,'turn_index':idx,'speaker_id':None,'role_inferido':'unknown','timestamp_raw':None,'timestamp_seconds':None,'turn_text_clean':txt,'speaker_raw':None,'turn_text_raw':txt})
                buffer=[]; n=0
            buffer.append(sent); n+=w
        if buffer:
            idx+=1
            txt=' '.join(buffer)
            rows.append({'doc_id':doc_id,'turn_index':idx,'speaker_id':None,'role_inferido':'unknown','timestamp_raw':None,'timestamp_seconds':None,'turn_text_clean':txt,'speaker_raw':None,'turn_text_raw':txt})
        return rows
    for i,m in enumerate(matches):
        start=m.end(); end=matches[i+1].start() if i+1<len(matches) else len(s)
        raw_text=s[start:end]
        txt=clean_text_for_embedding(raw_text)
        if len(txt.split())<2: continue
        raw_label=m.group(1)
        sp=re.search(r'\d+',raw_label or '')
        rows.append({'doc_id':doc_id,'turn_index':len(rows)+1,'speaker_id':int(sp.group()) if sp else None,'role_inferido':'unknown','timestamp_raw':m.group(2),'timestamp_seconds':parse_ts(m.group(2)),'turn_text_clean':txt,'speaker_raw':raw_label,'turn_text_raw':normalize_ws(raw_text)})
    return rows

def word_count(s):
    return len(re.findall(r'\b\w+\b', str(s), flags=re.UNICODE))

def is_question(s):
    return '?' in str(s) or bool(re.search(r'\b(o que|por que|porque|qual|quais|como|quando|onde|quem|voc[eê]|voces|vocês)\b',str(s).lower()))

def infer_roles(turns_doc):
    df=turns_doc.copy()
    if df.empty:
        return df
    df['n_words_turn']=df['turn_text_clean'].map(word_count)
    df['is_question_turn']=df['turn_text_clean'].map(is_question)
    # speaker stats
    if df['speaker_id'].notna().any():
        stats=df.groupby('speaker_id',dropna=True).agg(words=('n_words_turn','sum'), turns=('turn_index','count'), qrate=('is_question_turn','mean')).reset_index()
        total=max(stats['words'].sum(),1)
        stats['word_share']=stats['words']/total
        # top question rate/turns is likely interviewer, but be conservative
        interviewer=set(stats[(stats['qrate']>=0.45) & (stats['turns']>=5)]['speaker_id'].tolist())
        # if none, use most questions if high enough
        if not interviewer and len(stats)>1:
            cand=stats.sort_values(['qrate','turns'],ascending=False).iloc[0]
            if cand['qrate']>=0.35:
                interviewer={cand['speaker_id']}
        df['role_inferido']=df['speaker_id'].map(lambda x: 'interviewer_heuristic' if x in interviewer else 'participant_or_unknown')
    else:
        df['role_inferido']='unknown'
    return df

def language_scores(text):
    toks=re.findall(r'\b[a-záéíóúâêôãõçñü]+\b', str(text).lower())
    toks_norm=[strip_accents(t) for t in toks]
    n=max(len(toks_norm),1)
    c=Counter(toks_norm)
    pt=sum(c[w] for w in PT_LANG_WORDS)/n
    es=sum(c[w] for w in ES_LANG_WORDS)/n
    en=sum(c[w] for w in EN_LANG_WORDS)/n
    # accent signals
    pt += 0.01*len(re.findall(r'[ãõçâêô]',str(text).lower()))/max(len(text),1)*100
    es += 0.01*len(re.findall(r'[ñ¿¡]',str(text).lower()))/max(len(text),1)*100
    scores={'pt':pt,'es':es,'en':en}
    best=max(scores, key=scores.get)
    vals=sorted(scores.values(), reverse=True)
    margin=vals[0]-vals[1] if len(vals)>1 else vals[0]
    # If little evidence, use unknown. Mixed only if close between pt and es and enough evidence.
    if vals[0] < 0.025:
        label='unknown'
    elif margin < 0.015 and vals[1] > 0.035:
        label='mixed'
    else:
        label=best
    return label, scores, margin

def tokenize_keywords(text):
    s=strip_accents(str(text).lower())
    s=re.sub(r'\[[A-Z]+\]',' ',s)
    s=re.sub(r'[^a-z0-9çáéíóúâêôãõàüñ_\s-]',' ',s)
    s=re.sub(r'[_-]',' ',s)
    toks=re.findall(r'\b[a-záéíóúâêôãõçñü]{3,}\b',s)
    out=[]
    for t in toks:
        tn=strip_accents(t)
        if tn in STOPWORDS_KEYWORDS: continue
        if len(set(tn))<=2 and len(tn)>4: continue
        if tn.isdigit(): continue
        out.append(tn)
    return out

def procedural_score(text):
    toks=re.findall(r'\b\w+\b', str(text).lower())
    n=max(len(toks),1)
    matches=len(PROCEDURAL_RE.findall(str(text)))
    return matches, matches/max(n/80,1)

def build_units_from_turns(turns_doc):
    units=[]
    for _,r in turns_doc.iterrows():
        text=r['turn_text_clean']
        sents=split_sentences(text)
        current=[]; n=0
        for sent in sents:
            w=word_count(sent)
            if current and n+w > MAX_WORDS:
                txt=' '.join(current)
                units.append({**r.to_dict(),'unit_text':txt,'n_words_unit':word_count(txt)})
                current=[]; n=0
            current.append(sent); n+=w
        if current:
            txt=' '.join(current)
            units.append({**r.to_dict(),'unit_text':txt,'n_words_unit':word_count(txt)})
    return units

def segment_doc(doc_id, turns_doc):
    units=build_units_from_turns(turns_doc)
    chunks=[]; buf=[]; n=0; idx=0
    def flush():
        nonlocal buf,n,idx
        if not buf: return
        idx+=1
        text=' '.join([b['unit_text'] for b in buf])
        speaker_ids=sorted({int(b['speaker_id']) for b in buf if pd.notna(b.get('speaker_id'))})
        qshare=np.mean([bool(b.get('is_question_turn')) for b in buf]) if buf else 0
        interviewer_words=sum(b.get('n_words_unit',0) for b in buf if b.get('role_inferido')=='interviewer_heuristic')
        timestamps=[b.get('timestamp_seconds') for b in buf if b.get('timestamp_seconds') is not None and not pd.isna(b.get('timestamp_seconds'))]
        proc_matches, proc_score=procedural_score(text)
        chunks.append({
            'doc_id':doc_id,
            'chunk_index':idx,
            'text_for_embedding':clean_text_for_embedding(text),
            'turn_index_start':min([int(b['turn_index']) for b in buf]),
            'turn_index_end':max([int(b['turn_index']) for b in buf]),
            'speaker_ids':speaker_ids,
            'n_speakers':len(speaker_ids),
            'question_unit_share':float(qshare),
            'interviewer_word_share_heuristic':float(interviewer_words/max(n,1)),
            'timestamp_start_seconds':min(timestamps) if timestamps else None,
            'timestamp_end_seconds':max(timestamps) if timestamps else None,
            'procedural_matches':proc_matches,
            'procedural_score':proc_score,
        })
        buf=[]; n=0
    for u in units:
        w=u['n_words_unit']
        if buf and (n+w>MAX_WORDS) and n>=MIN_WORDS:
            flush()
        buf.append(u); n+=w
        if n>=TARGET_WORDS:
            flush()
    if buf:
        # merge tiny tail with previous if possible
        tail_words=n
        if chunks and tail_words<MIN_WORDS:
            prev=chunks.pop()
            # reconstruct using previous text as a pseudo unit (simpler: append tail text)
            combined_text=(prev['text_for_embedding']+' '+' '.join([b['unit_text'] for b in buf])).strip()
            idx=prev['chunk_index']-1
            buf_prev=[{'unit_text':combined_text,'speaker_id':None,'is_question_turn':False,'role_inferido':'unknown','n_words_unit':word_count(combined_text),'turn_index':prev['turn_index_start'],'timestamp_seconds':prev['timestamp_start_seconds']}]
            buf=buf_prev
            n=word_count(combined_text)
            flush()
        else:
            flush()
    for c in chunks:
        c['chunk_id']=f"{doc_id}_pp_ch_{c['chunk_index']:04d}"
    return chunks

Carregando snapshot local: /Users/emanuelgandra/Desktop/Faculdade/6Período/ProjetoCienciaDados/Kyra/kyrav2/data/raw/bq_snapshots/kyra_pml_20261__kyra_base_2026_1__entrevistas_docs_v2.parquet
RUN_ID 20260520_111127
PROJECT_DIR /Users/emanuelgandra/Desktop/Faculdade/6Período/ProjetoCienciaDados/Kyra/kyrav2
docs_raw (155, 29)


## 1. Pré-processamento no nível documento

In [2]:
# docs preprocessed
records=[]
for _,r in docs_raw.iterrows():
    raw=normalize_ws(r.get(TEXT_COL,''))
    pre=clean_pre_segmentation(raw)
    clean_embed=clean_text_for_embedding(pre)
    rec={'doc_id':r[DOC_ID_COL], 'idioma_original':r.get(LANG_COL,None), 'texto_raw':raw, 'texto_clean':clean_embed,
         'n_words_raw':word_count(raw), 'n_words_clean':word_count(clean_embed),
         'clean_word_retention': word_count(clean_embed)/max(word_count(raw),1),
         'n_chars_raw':len(raw), 'n_chars_clean':len(clean_embed), 'doc_text_hash_clean':md5_short(clean_embed)}
    for col in docs_raw.columns:
        if col not in [TEXT_COL,DOC_ID_COL,LANG_COL] and col not in rec:
            rec[col]=r.get(col)
    records.append(rec)
docs_pre=pd.DataFrame(records)

## 2. Extração dos turnos/falas

In [3]:
# turns
turn_rows=[]
for _,r in docs_raw.iterrows():
    t=pd.DataFrame(extract_turns_for_doc(r[DOC_ID_COL], r[TEXT_COL]))
    if not t.empty:
        t=infer_roles(t)
        turn_rows.extend(t.to_dict('records'))
turns_pre=pd.DataFrame(turn_rows)
turns_pre['n_words_turn']=turns_pre['turn_text_clean'].map(word_count)
turns_pre['is_question_turn']=turns_pre['turn_text_clean'].map(is_question)
turns_pre['n_chars_turn']=turns_pre['turn_text_clean'].str.len()
turns_pre['turn_hash']=turns_pre['turn_text_clean'].map(md5_short)
# doc speaker word share
if turns_pre['speaker_id'].notna().any():
    totals=turns_pre.groupby('doc_id')['n_words_turn'].transform('sum').replace(0,1)
    sp_totals=turns_pre.groupby(['doc_id','speaker_id'])['n_words_turn'].transform('sum')
    turns_pre['speaker_word_share_doc']=sp_totals/totals
else:
    turns_pre['speaker_word_share_doc']=np.nan

## 3. Segmentação em chunks

In [4]:
# chunks
chunks=[]
for doc_id,td in turns_pre.groupby('doc_id', sort=False):
    chunks.extend(segment_doc(doc_id, td.sort_values('turn_index')))
chunks_pre=pd.DataFrame(chunks)

## 4. Enriquecimento dos chunks

In [5]:
# enrich chunks
for col in ['source_filename_hash','collection_batch','mes','mes_num','ano','projeto','pais','marca_foco','publico','tipo_sessao','canal_origem','contexto_origem','idioma_original','data_bruta_nome_arquivo','hora_bruta_nome_arquivo','session_prefix','session_num','status_anonimizacao','status_extracao','qualidade_texto_global','requires_manual_review','duracao_aprox_min','resumo_executivo_curto','evidencia_citavel_doc']:
    if col in docs_pre.columns:
        chunks_pre=chunks_pre.merge(docs_pre[['doc_id',col]],on='doc_id',how='left')

## 5. Features, idioma, keywords e filtros

In [6]:
# features
langs=[]; scs=[]; margins=[]; tokens=[]
for text in chunks_pre['text_for_embedding']:
    lang, scores, margin=language_scores(text)
    langs.append(lang); scs.append(scores); margins.append(margin); tokens.append(tokenize_keywords(text))
chunks_pre['idioma_detectado']=langs
chunks_pre['score_pt']=[s['pt'] for s in scs]; chunks_pre['score_es']=[s['es'] for s in scs]; chunks_pre['score_en']=[s['en'] for s in scs]
chunks_pre['lang_margin']=margins
chunks_pre['tokens_topic']=tokens
chunks_pre['text_for_keywords']=[' '.join(t) for t in tokens]
chunks_pre['n_words']=chunks_pre['text_for_embedding'].map(word_count)
chunks_pre['n_chars']=chunks_pre['text_for_embedding'].str.len()
chunks_pre['n_topic_tokens']=chunks_pre['tokens_topic'].map(len)
chunks_pre['lexical_diversity_topic']=chunks_pre['tokens_topic'].map(lambda x: len(set(x))/max(len(x),1))
chunks_pre['chunk_text_hash_clean']=chunks_pre['text_for_embedding'].map(md5_short)
chunks_pre['chunk_text_hash_keywords']=chunks_pre['text_for_keywords'].map(md5_short)
chunks_pre['alpha_ratio']=chunks_pre['text_for_embedding'].map(lambda s: sum(ch.isalpha() for ch in str(s))/max(len(str(s)),1))
chunks_pre['is_exact_duplicate_clean']=chunks_pre.duplicated('chunk_text_hash_clean', keep='first')
# residual flags
chunks_pre['has_page_counter_residual']=chunks_pre['text_for_embedding'].str.contains(PAGE_COUNTER_INLINE_RE, regex=True, na=False)
chunks_pre['has_filename_residual']=chunks_pre['text_for_embedding'].str.contains(r'\.(m4a|mp3|mp4|wav|pdf)\b|\b[A-Z]{2}_\d{4}_', regex=True, case=False, na=False)
chunks_pre['has_meeting_meta_residual']=chunks_pre['text_for_embedding'].str.contains(r'Meeting Title|Meeting created at|Transcribed URL|Transcribed by|Recording URL|Download URL|\[URL\]|https?://', regex=True, case=False, na=False)
chunks_pre['has_timestamp_residual']=chunks_pre['text_for_embedding'].str.contains(TIMESTAMP_RE, regex=True, na=False)
chunks_pre['has_noise_residual']=chunks_pre[['has_page_counter_residual','has_filename_residual','has_meeting_meta_residual','has_timestamp_residual']].any(axis=1)
chunks_pre['is_language_pt']=chunks_pre['idioma_detectado'].eq('pt')
chunks_pre['is_language_es']=chunks_pre['idioma_detectado'].eq('es')
chunks_pre['is_language_allowed']=chunks_pre['is_language_pt']
# quality reasons
quality=[]
for _,r in chunks_pre.iterrows():
    reasons=[]
    if r['n_words']<MIN_WORDS: reasons.append('too_short')
    if r['n_topic_tokens']<MIN_TOPIC_TOKENS: reasons.append('few_topic_tokens')
    if r['alpha_ratio']<0.70: reasons.append('low_alpha_ratio')
    if r['is_exact_duplicate_clean']: reasons.append('exact_duplicate_clean')
    if r['has_noise_residual']: reasons.append('metadata_noise_residual')
    if r['procedural_score']>=1.0 or (r['procedural_matches']>=3 and r['n_topic_tokens']<60): reasons.append('procedural_intro_outro_or_consent')
    if not r['is_language_allowed']: reasons.append(f"language_not_{ALLOWED_LANG}")
    quality.append(reasons)
chunks_pre['quality_reasons']=quality
chunks_pre['quality_reasons_str']=chunks_pre['quality_reasons'].map(lambda x: ';'.join(x))
chunks_pre['keep_for_modeling_all_quality']=chunks_pre['quality_reasons'].map(lambda x: not any(reason not in [f'language_not_{ALLOWED_LANG}'] for reason in x))
chunks_pre['keep_for_modeling']=chunks_pre['quality_reasons'].map(len).eq(0)
chunks_pre['quality_status']=np.select([
    chunks_pre['keep_for_modeling'],
    chunks_pre['quality_reasons'].map(lambda xs: any(r.startswith('language_not_') for r in xs) and len(xs)==1),
],['ok','excluded_by_language_only'], default='excluded_quality')
chunks_pre['texto_preprocessado']=chunks_pre['text_for_embedding']
chunks_pre['source_segmentation']='full_interview_turn_sentence_v2_cleaned_before_segmentation'
chunks_pre['preprocess_run_id']=RUN_ID
# ordering columns
front=['doc_id','chunk_id','chunk_index','keep_for_modeling','quality_status','quality_reasons_str','idioma_detectado','score_pt','score_es','score_en','text_for_embedding','text_for_keywords','tokens_topic','n_words','n_topic_tokens','n_chars','lexical_diversity_topic']
chunks_pre=chunks_pre[[c for c in front if c in chunks_pre.columns]+[c for c in chunks_pre.columns if c not in front]]

/var/folders/11/lm50x6ys7ng_15t73h0j41fm0000gn/T/ipykernel_28359/3876725189.py:21: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  chunks_pre['has_filename_residual']=chunks_pre['text_for_embedding'].str.contains(r'\.(m4a|mp3|mp4|wav|pdf)\b|\b[A-Z]{2}_\d{4}_', regex=True, case=False, na=False)


## 6. Bases finais por idioma

In [7]:
# modeling bases
chunks_modeling_all=chunks_pre[chunks_pre['keep_for_modeling_all_quality']].copy()
chunks_modeling_pt=chunks_pre[chunks_pre['keep_for_modeling'] & chunks_pre['idioma_detectado'].eq('pt')].copy()
chunks_modeling_es=chunks_pre[chunks_pre['keep_for_modeling_all_quality'] & chunks_pre['idioma_detectado'].eq('es')].copy()
chunks_modeling_mixed=chunks_pre[chunks_pre['keep_for_modeling_all_quality'] & chunks_pre['idioma_detectado'].eq('mixed')].copy()

## 7. Tokens long alinhados à modelagem

In [8]:
# tokens long
def tokens_long_from_chunks(df):
    rows=[]
    for _,r in df[['doc_id','chunk_id','chunk_index','tokens_topic']].iterrows():
        counts=Counter(r['tokens_topic'])
        for tok,freq in counts.items():
            rows.append({'doc_id':r['doc_id'],'chunk_id':r['chunk_id'],'chunk_index':r['chunk_index'],'token':tok,'freq_in_chunk':freq})
    return pd.DataFrame(rows)
tokens_long_all=tokens_long_from_chunks(chunks_pre)
tokens_long_modeling=tokens_long_from_chunks(chunks_modeling_pt)
print('docs_pre',docs_pre.shape,'turns',turns_pre.shape,'chunks',chunks_pre.shape,'model pt',chunks_modeling_pt.shape,'allq',chunks_modeling_all.shape,'esq',chunks_modeling_es.shape,'tokens_model',tokens_long_modeling.shape)
print('res noise pt', chunks_modeling_pt[['has_page_counter_residual','has_filename_residual','has_meeting_meta_residual']].sum())
print('docs in pt modeling',chunks_modeling_pt.doc_id.nunique())
print('lang dist pre', chunks_pre.idioma_detectado.value_counts())
print('top tokens model', tokens_long_modeling.groupby('token').freq_in_chunk.sum().sort_values(ascending=False).head(20).to_string())

docs_pre (155, 36) turns (63647, 14) chunks (6228, 69) model pt (4976, 69) allq (6055, 69) esq (1079, 69) tokens_model (334162, 5)
res noise pt has_page_counter_residual    0
has_filename_residual        0
has_meeting_meta_residual    0
dtype: int64
docs in pt modeling 131
lang dist pre idioma_detectado
pt    5149
es    1079
Name: count, dtype: int64
top tokens model token
natura       5560
produto      4607
pele         3194
produtos     2704
pode         2427
sempre       2141
ver          2119
marca        1960
presente     1932
comprar      1886
embalagem    1858
exemplo      1825
boticario    1785
nome         1751
legal        1738
questao      1683
tempo        1646
loja         1641
gosto        1632
fica         1624


## 8. Salvamento e diagnósticos

In [9]:
# save

# Para evitar diagnósticos velhos de execuções anteriores.
for stale in [
    '00_preprocess_summary_bases.csv',
    '01_language_distribution_chunks.csv',
    '01_speaker_role_summary.csv',
    '02_quality_reasons_counts.csv',
    '02_tokens_limpos_top3000.csv',
    '03_residual_noise_modeling_pt.csv',
    '04_segmentacao_por_documento.csv',
    '05_top_tokens_modeling_pt.csv',
    '06_tfidf_por_projeto_top50.csv',
    '06_checklist_final.csv',
    '07_comparacao_segmentacao_antiga_vs_nova.csv',
    '08_preprocess_summary.json',
    'README_preprocessamento.md',
    'preprocess_resumo.md',
    'chunks_excluidos_amostra1000.csv',
]:
    p = DIAG_DIR / stale
    if p.exists():
        p.unlink()

# Bases principais e auxiliares.
outputs_to_save = [
    ('interviews_docs_preprocessed.parquet', docs_pre),
    ('interviews_turns_preprocessed.parquet', turns_pre),
    ('interviews_chunks_preprocessed.parquet', chunks_pre),
    ('interviews_chunks_modeling_all.parquet', chunks_modeling_all),
    ('interviews_chunks_modeling_pt.parquet', chunks_modeling_pt),
    ('interviews_chunks_modeling_es.parquet', chunks_modeling_es),
    ('interviews_chunks_modeling_mixed.parquet', chunks_modeling_mixed),
    ('interviews_chunks_modeling.parquet', chunks_modeling_pt),  # base principal por enquanto: somente PT
    ('interviews_tokens_long_all.parquet', tokens_long_all),
    ('interviews_tokens_long_modeling.parquet', tokens_long_modeling),
    ('interviews_tokens_long.parquet', tokens_long_modeling),      # alinhada com a principal
]
for name, df in outputs_to_save:
    out = PROCESSED_DIR / name
    df.to_parquet(out, index=False)
    print('salvo:', out, df.shape)

# CSVs leves para inspeção rápida.
chunks_modeling_pt.head(1000).to_csv(PROCESSED_DIR / 'interviews_chunks_modeling_pt_amostra1000.csv', index=False, encoding='utf-8-sig')
chunks_pre[~chunks_pre['keep_for_modeling']].head(1000).to_csv(DIAG_DIR / 'chunks_excluidos_amostra1000.csv', index=False, encoding='utf-8-sig')

# Diagnósticos finais.
summary_bases = pd.DataFrame([
    {'base': 'docs_pre', 'linhas': len(docs_pre), 'docs_unicos': docs_pre['doc_id'].nunique()},
    {'base': 'turns_pre', 'linhas': len(turns_pre), 'docs_unicos': turns_pre['doc_id'].nunique()},
    {'base': 'chunks_pre', 'linhas': len(chunks_pre), 'docs_unicos': chunks_pre['doc_id'].nunique()},
    {'base': 'chunks_modeling_all_quality', 'linhas': len(chunks_modeling_all), 'docs_unicos': chunks_modeling_all['doc_id'].nunique()},
    {'base': 'chunks_modeling_pt/principal', 'linhas': len(chunks_modeling_pt), 'docs_unicos': chunks_modeling_pt['doc_id'].nunique()},
    {'base': 'chunks_modeling_es', 'linhas': len(chunks_modeling_es), 'docs_unicos': chunks_modeling_es['doc_id'].nunique()},
    {'base': 'chunks_modeling_mixed', 'linhas': len(chunks_modeling_mixed), 'docs_unicos': chunks_modeling_mixed['doc_id'].nunique()},
    {'base': 'tokens_long_modeling', 'linhas': len(tokens_long_modeling), 'docs_unicos': tokens_long_modeling['doc_id'].nunique()},
])
summary_bases.to_csv(DIAG_DIR / '00_preprocess_summary_bases.csv', index=False, encoding='utf-8-sig')

chunks_pre['idioma_detectado'].value_counts(dropna=False).rename_axis('idioma_detectado').reset_index(name='n_chunks').to_csv(
    DIAG_DIR / '01_language_distribution_chunks.csv', index=False, encoding='utf-8-sig'
)

# Speaker summary.
if {'doc_id', 'speaker_id', 'role_inferido', 'n_words_turn', 'is_question_turn', 'speaker_word_share_doc'}.issubset(turns_pre.columns):
    speaker_role_summary = (
        turns_pre.dropna(subset=['speaker_id'])
        .groupby(['doc_id', 'speaker_id', 'role_inferido'], dropna=False)
        .agg(
            n_turns=('turn_index', 'count'),
            n_words=('n_words_turn', 'sum'),
            question_rate=('is_question_turn', 'mean'),
            word_share=('speaker_word_share_doc', 'max'),
        )
        .reset_index()
    )
    speaker_role_summary.to_csv(DIAG_DIR / '01_speaker_role_summary.csv', index=False, encoding='utf-8-sig')

quality_counts = chunks_pre['quality_reasons'].explode().dropna().value_counts().rename_axis('reason').reset_index(name='n_chunks')
quality_counts.to_csv(DIAG_DIR / '02_quality_reasons_counts.csv', index=False, encoding='utf-8-sig')

# Top tokens já alinhados à base principal PT.
top_tokens_modeling = (
    tokens_long_modeling.groupby('token', as_index=False)['freq_in_chunk']
    .sum()
    .sort_values('freq_in_chunk', ascending=False)
)
top_tokens_modeling.rename(columns={'freq_in_chunk': 'freq'}).head(3000).to_csv(
    DIAG_DIR / '02_tokens_limpos_top3000.csv', index=False, encoding='utf-8-sig'
)
top_tokens_modeling.head(1000).to_csv(DIAG_DIR / '05_top_tokens_modeling_pt.csv', index=False, encoding='utf-8-sig')

noise_cols = ['has_page_counter_residual', 'has_filename_residual', 'has_meeting_meta_residual', 'has_timestamp_residual']
residual_noise_modeling = chunks_modeling_pt[noise_cols].sum().rename('n_chunks').reset_index().rename(columns={'index': 'flag'})
residual_noise_modeling.to_csv(DIAG_DIR / '03_residual_noise_modeling_pt.csv', index=False, encoding='utf-8-sig')

segmentacao_por_doc = (
    chunks_pre.groupby('doc_id')
    .agg(
        n_chunks=('chunk_id', 'count'),
        n_chunks_modeling_pt=('keep_for_modeling', 'sum'),
        n_words_mean=('n_words', 'mean'),
        n_words_min=('n_words', 'min'),
        n_words_max=('n_words', 'max'),
        noise_residual_chunks=('has_noise_residual', 'sum'),
    )
    .reset_index()
)
segmentacao_por_doc.to_csv(DIAG_DIR / '04_segmentacao_por_documento.csv', index=False, encoding='utf-8-sig')

# TF-IDF por projeto, usando a coluna limpa para keywords.
try:
    from sklearn.feature_extraction.text import TfidfVectorizer
    if 'projeto' in chunks_modeling_pt.columns and chunks_modeling_pt['projeto'].notna().any():
        tfidf_rows = []
        for projeto, g in chunks_modeling_pt.groupby('projeto', dropna=False):
            texts = g['text_for_keywords'].fillna('').astype(str).tolist()
            texts = [t for t in texts if t.strip()]
            if len(texts) < 2:
                continue
            vec = TfidfVectorizer(min_df=1, max_df=0.90, ngram_range=(1, 2), token_pattern=r'(?u)\b[a-zA-ZÀ-ÿ]{3,}\b')
            X = vec.fit_transform(texts)
            terms = np.array(vec.get_feature_names_out())
            means = np.asarray(X.mean(axis=0)).ravel()
            order = means.argsort()[::-1][:50]
            for i in order:
                tfidf_rows.append({'projeto': projeto, 'termo': terms[i], 'mean_tfidf': float(means[i])})
        pd.DataFrame(tfidf_rows).to_csv(DIAG_DIR / '06_tfidf_por_projeto_top50.csv', index=False, encoding='utf-8-sig')
except Exception as exc:
    print('Aviso: não foi possível gerar TF-IDF por projeto:', exc)

# Comparação com chunks antigos, se o snapshot antigo existir/acessar.
try:
    old_raw = load_table_or_snapshot(OLD_CHUNKS_TABLE)
    if 'doc_id' in old_raw.columns:
        old_text_col = 'texto_analise' if 'texto_analise' in old_raw.columns else None
        if old_text_col:
            old_raw['_old_words'] = old_raw[old_text_col].fillna('').astype(str).map(word_count)
            old_summary = old_raw.groupby('doc_id').agg(
                old_n_chunks=('_old_words', 'count'),
                old_mean_words=('_old_words', 'mean'),
                old_p50_words=('_old_words', 'median'),
            ).reset_index()
            new_summary = chunks_pre.groupby('doc_id').agg(
                new_n_chunks=('chunk_id', 'count'),
                new_mean_words=('n_words', 'mean'),
                new_p50_words=('n_words', 'median'),
                new_kept_chunks=('keep_for_modeling', 'sum'),
            ).reset_index()
            comp = old_summary.merge(new_summary, on='doc_id', how='outer')
            comp['delta_n_chunks_new_minus_old'] = comp['new_n_chunks'] - comp['old_n_chunks']
            comp['ratio_new_old_chunks'] = comp['new_n_chunks'] / comp['old_n_chunks'].replace(0, np.nan)
            comp.to_csv(DIAG_DIR / '07_comparacao_segmentacao_antiga_vs_nova.csv', index=False, encoding='utf-8-sig')
except Exception as exc:
    print('Aviso: comparação com chunks antigos não foi gerada:', exc)

checks = pd.DataFrame([
    {'check': 'doc_id unico em docs_pre', 'ok': bool(docs_pre['doc_id'].is_unique), 'valor': docs_pre['doc_id'].nunique()},
    {'check': 'chunk_id unico em chunks_pre', 'ok': bool(chunks_pre['chunk_id'].is_unique), 'valor': chunks_pre['chunk_id'].nunique()},
    {'check': 'chunk_id unico em chunks_modeling_pt', 'ok': bool(chunks_modeling_pt['chunk_id'].is_unique), 'valor': chunks_modeling_pt['chunk_id'].nunique()},
    {'check': 'tokens_long_modeling alinhado com chunks_modeling_pt', 'ok': set(tokens_long_modeling['chunk_id']) == set(chunks_modeling_pt['chunk_id']), 'valor': f"{tokens_long_modeling['chunk_id'].nunique()} vs {chunks_modeling_pt['chunk_id'].nunique()}"},
    {'check': 'sem page counters na base principal', 'ok': int(chunks_modeling_pt['has_page_counter_residual'].sum()) == 0, 'valor': int(chunks_modeling_pt['has_page_counter_residual'].sum())},
    {'check': 'sem filenames/metadados na base principal', 'ok': int(chunks_modeling_pt[['has_filename_residual','has_meeting_meta_residual']].any(axis=1).sum()) == 0, 'valor': int(chunks_modeling_pt[['has_filename_residual','has_meeting_meta_residual']].any(axis=1).sum())},
    {'check': 'sem timestamps na base principal', 'ok': int(chunks_modeling_pt['has_timestamp_residual'].sum()) == 0, 'valor': int(chunks_modeling_pt['has_timestamp_residual'].sum())},
    {'check': 'base principal somente pt', 'ok': set(chunks_modeling_pt['idioma_detectado'].unique()).issubset({'pt'}), 'valor': ','.join(sorted(chunks_modeling_pt['idioma_detectado'].unique()))},
])
checks.to_csv(DIAG_DIR / '06_checklist_final.csv', index=False, encoding='utf-8-sig')

summary_json = {
    'run_id': RUN_ID,
    'n_docs': int(len(docs_pre)),
    'n_turns': int(len(turns_pre)),
    'n_new_chunks_total': int(len(chunks_pre)),
    'n_new_chunks_modeling_pt': int(len(chunks_modeling_pt)),
    'n_docs_modeling_pt': int(chunks_modeling_pt['doc_id'].nunique()),
    'pct_chunks_kept_for_modeling_pt': float(len(chunks_modeling_pt) / max(len(chunks_pre), 1)),
    'mean_words_new_chunks': float(chunks_pre['n_words'].mean()),
    'median_words_new_chunks': float(chunks_pre['n_words'].median()),
    'mean_topic_tokens_modeling_pt': float(chunks_modeling_pt['n_topic_tokens'].mean()) if len(chunks_modeling_pt) else 0.0,
    'language_distribution_all_chunks': {str(k): int(v) for k, v in chunks_pre['idioma_detectado'].value_counts(dropna=False).to_dict().items()},
    'language_distribution_modeling_pt': {str(k): int(v) for k, v in chunks_modeling_pt['idioma_detectado'].value_counts(dropna=False).to_dict().items()},
    'quality_reason_counts': {str(k): int(v) for k, v in chunks_pre['quality_reasons'].explode().dropna().value_counts().to_dict().items()},
    'residual_noise_modeling_pt': {row['flag']: int(row['n_chunks']) for _, row in residual_noise_modeling.iterrows()},
    'checklist_all_ok': bool(checks['ok'].all()),
    'outputs': {
        'main_modeling': str(PROCESSED_DIR / 'interviews_chunks_modeling.parquet'),
        'modeling_pt': str(PROCESSED_DIR / 'interviews_chunks_modeling_pt.parquet'),
        'modeling_all': str(PROCESSED_DIR / 'interviews_chunks_modeling_all.parquet'),
        'tokens_long_modeling': str(PROCESSED_DIR / 'interviews_tokens_long_modeling.parquet'),
    },
}
with open(DIAG_DIR / '08_preprocess_summary.json', 'w', encoding='utf-8') as f:
    json.dump(summary_json, f, ensure_ascii=False, indent=2)

params = {
    'run_id': RUN_ID,
    'docs_table': DOCS_TABLE,
    'old_chunks_table_reference_only': OLD_CHUNKS_TABLE,
    'modeling_language': ALLOWED_LANG,
    'target_words': TARGET_WORDS,
    'min_words': MIN_WORDS,
    'max_words': MAX_WORDS,
    'hard_max_words': HARD_MAX_WORDS,
    'min_topic_tokens': MIN_TOPIC_TOKENS,
    'outputs': summary_json['outputs'],
}
with open(PROCESSED_DIR / 'preprocess_params.json', 'w', encoding='utf-8') as f:
    json.dump(params, f, ensure_ascii=False, indent=2)

summary_md = f"""# Pré-processamento de entrevistas KYRA

Run ID: `{RUN_ID}`

## Decisão metodológica

A segmentação final foi reconstruída a partir da tabela de entrevistas completas, não da tabela antiga de chunks.
A tabela antiga é usada apenas para comparação/auditoria.

## Base principal

Por enquanto, a base principal contém somente chunks classificados como português:

- `data/processed/interviews_chunks_modeling.parquet`
- `data/processed/interviews_chunks_modeling_pt.parquet`

## Volumes

- Entrevistas: {len(docs_pre):,}
- Turnos extraídos: {len(turns_pre):,}
- Novos chunks totais: {len(chunks_pre):,}
- Novos chunks PT para modelagem: {len(chunks_modeling_pt):,}
- Docs com chunks PT para modelagem: {chunks_modeling_pt['doc_id'].nunique():,}
- Percentual mantido na base PT: {len(chunks_modeling_pt) / max(len(chunks_pre), 1):.1%}
- Mediana de palavras por novo chunk: {chunks_pre['n_words'].median():.1f}
- Média de tokens explicativos por chunk PT: {chunks_modeling_pt['n_topic_tokens'].mean() if len(chunks_modeling_pt) else 0:.1f}

## Checklist

{checks.to_markdown(index=False)}

## Ruído residual na base principal PT

{residual_noise_modeling.to_markdown(index=False)}

## Uso recomendado

Para embeddings/clusterização, use:

- ID: `chunk_id`
- Texto: `text_for_embedding`
- Keywords explicativas: `text_for_keywords`
- Metadados: `doc_id`, `projeto`, `marca_foco`, `publico`, `tipo_sessao`, se existirem.

Não use stemming como entrada principal de embeddings.
"""
(DIAG_DIR / 'README_preprocessamento.md').write_text(summary_md, encoding='utf-8')
(DIAG_DIR / 'preprocess_resumo.md').write_text(summary_md, encoding='utf-8')

print('\nResumo das bases:')
print(summary_bases.to_string(index=False))
print('\nChecklist final:')
print(checks.to_string(index=False))
print('\nBase principal para embeddings/clusters:', PROCESSED_DIR / 'interviews_chunks_modeling.parquet')
print('Diagnósticos:', DIAG_DIR)


salvo: /Users/emanuelgandra/Desktop/Faculdade/6Período/ProjetoCienciaDados/Kyra/kyrav2/data/processed/interviews_docs_preprocessed.parquet (155, 36)
salvo: /Users/emanuelgandra/Desktop/Faculdade/6Período/ProjetoCienciaDados/Kyra/kyrav2/data/processed/interviews_turns_preprocessed.parquet (63647, 14)
salvo: /Users/emanuelgandra/Desktop/Faculdade/6Período/ProjetoCienciaDados/Kyra/kyrav2/data/processed/interviews_chunks_preprocessed.parquet (6228, 69)
salvo: /Users/emanuelgandra/Desktop/Faculdade/6Período/ProjetoCienciaDados/Kyra/kyrav2/data/processed/interviews_chunks_modeling_all.parquet (6055, 69)
salvo: /Users/emanuelgandra/Desktop/Faculdade/6Período/ProjetoCienciaDados/Kyra/kyrav2/data/processed/interviews_chunks_modeling_pt.parquet (4976, 69)
salvo: /Users/emanuelgandra/Desktop/Faculdade/6Período/ProjetoCienciaDados/Kyra/kyrav2/data/processed/interviews_chunks_modeling_es.parquet (1079, 69)
salvo: /Users/emanuelgandra/Desktop/Faculdade/6Período/ProjetoCienciaDados/Kyra/kyrav2